# Mitigating Multimodal Shortcut Learning via Diagnostic Selective Routing
**Cuaderno Experimental y Auditoría de Seguridad (6 ECTS — Multimodal Trust & Safety)**

---

### Arquitectura de Ejecución Desacoplada
* **Scripts de Cómputo (`src/`):** Ejecutan extracción offline (`fp16`), entrenamiento del *Fusion Zoo* y evaluación contrafactual en GPU (8 GB VRAM).
* **Artefactos (`results/`, `models/`, `logs/`):** Persisten métricas en JSON, checkpoints `.pt` y trazas de auditoría `JSONL`.
* **Este Notebook (`notebooks/`):** Orquesta las llamadas, valida integridad de datos, genera figuras vectoriales (300 dpi) para el paper y audita el comportamiento del *Human-in-the-Loop (HITL)*.

## Fase 0: Configuración de Entorno y Verificación de Hardware
**Objetivo:** Asegurar determinismo experimental (semillas fijas), comprobar disponibilidad de CUDA en la torre y verificar la jerarquía de directorios.

In [ ]:
import os, sys, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Asegurar raíz del proyecto en PYTHONPATH
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Directorio base: {PROJECT_ROOT}')
print(f'Dispositivo activo: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

## Fase 1: Ingeniería de Datos y Aislamiento de *Benign Confounders*
**Fundamento Científico:** Evaluar únicamente sobre el conjunto global enmascara el *colapso unimodal*. Para demostrar rigor en visión por computador, aislamos pares mínimos contrafactuales en `dev` (mismo texto con distinta imagen e inversión de etiqueta) y auditamos la distribución de clases.

**Scripts ejecutados en torre:**
```bash
PYTHONPATH=. python3 src/data/extract_text_features.py
PYTHONPATH=. python3 src/data/extract_visual_features.py
```

In [ ]:
# Inspección del dataset y del split contrafactual aislado
raw_dir = Path('data/raw')
dev_file = raw_dir / 'dev.jsonl' if (raw_dir / 'dev.jsonl').exists() else raw_dir / 'dev_seen.jsonl'
train_records = [json.loads(l) for l in open(raw_dir / 'train.jsonl', encoding='utf-8')]
dev_records = [json.loads(l) for l in open(dev_file, encoding='utf-8')]

conf_path = Path('data/processed/dev_confounders.json')
conf_indices = json.load(open(conf_path)) if conf_path.exists() else []

print(f'Train samples: {len(train_records)} | Positivos (Hate): {sum(r["label"] for r in train_records)/len(train_records):.2%}')
print(f'Dev samples:   {len(dev_records)} | Positivos (Hate): {sum(r["label"] for r in dev_records)/len(dev_records):.2%}')
print(f'Confounders contrafactuales aislados en Dev: {len(conf_indices)} muestras ({len(conf_indices)/len(dev_records):.1%})')
print(f'Tensores cacheados en data/features/: {len(list(Path("data/features").glob("*.pt")))} archivos')

## Fase 2: Estudio de Ablación del *Fusion Zoo* (5 Arquitecturas)
**Hipótesis:** La concatenación tardía (`concat`) sufre dominancia textual, produciendo una alta tasa de falsos positivos (FPR) sobre *benign confounders* y una brecha mínima respecto al modelo puramente textual ($|p_{\text{joint}} - p_{\text{text}}| \to 0$). Las interacciones multiplicativas (`hadamard`) y la atención cruzada espacial con `CoordConv` y *Modality Dropout* (`cross_attn`) fuerzan la dependencia visual.

**Script ejecutado en torre:**
```bash
PYTHONPATH=. python3 src/training/run_experiments.py
```

In [ ]:
summary_path = Path('results/metrics/ablation_summary.json')
if summary_path.exists():
    ablation_data = json.load(open(summary_path))
    df_ablation = pd.DataFrame(ablation_data).T
    df_ablation.index.name = 'Architecture'
    display(df_ablation)
else:
    print('Pendiente de ejecutar src/training/run_experiments.py en la torre.')

## Fase 3: Predicción Selectiva Diagnóstica y Curva *Risk-Coverage* (HITL)
**Novedad Metodológica:** A diferencia de los routers tradicionales basados en un umbral escalar de confianza ($\hat{p} \approx 0.5$), nuestro `SelectiveSafetyRouter` combina:
1. **Incertidumbre Epistémica ($\sigma^2_{\text{MC}}$):** Varianza tras $K=5$ pasadas estocásticas con MC-Dropout.
2. **Sensibilidad Visual Contrafactual ($\Delta_{\text{vis}}$):** Caída de probabilidad al anular el tensor espacial de imagen ($|p(\mathbf{v}, \mathbf{t}) - p(\mathbf{0}, \mathbf{t})|$).

Aquí evaluamos cómo disminuye el riesgo (error y FPR en confounders) a medida que derivamos a revisión humana (`FLAG_FOR_HUMAN_REVIEW`) los casos sospechosos.

In [ ]:
from src.models.fusion_zoo import MultimodalFusionZoo
from src.utils.selective_router import SelectiveSafetyRouter
from src.training.run_experiments import CachedMultimodalDataset

weights_file = Path('models/weights/cross_attn_best.pt')
if weights_file.exists():
    model = MultimodalFusionZoo(arch='cross_attn').to(device)
    model.load_state_dict(torch.load(weights_file, map_location=device, weights_only=True))
    router = SelectiveSafetyRouter(model, var_threshold=0.015, visual_delta_threshold=0.12)

    dev_ds = CachedMultimodalDataset(str(dev_file))
    audit_records = []
    for idx in range(len(dev_ds)):
        sample = dev_ds[idx]
        res = router.route(sample['image_feat'].unsqueeze(0).to(device), sample['text'].unsqueeze(0).to(device))
        res['target'] = int(sample['label'].item())
        res['is_confounder'] = idx in set(conf_indices)
        audit_records.append(res)

    df_audit = pd.DataFrame(audit_records)
    print('Distribución de decisiones del Router HITL:')
    print(df_audit['action'].value_counts(normalize=True).round(4))
    print('\nMotivos de derivación humana:')
    print(df_audit['reason'].value_counts(dropna=True))
else:
    print('Pendiente de generar models/weights/cross_attn_best.pt')

## Fase 4: Auditoría Espacial con Grad-CAM Determinista y Red Teaming
**Verificación de Seguridad:**
1. **Red Teaming Sintético (Modal Ablation Attack):** Inyectamos una imagen nula (`torch.zeros`) con textos altamente tóxicos para verificar que el sistema intercepta el atajo textual (`UNIMODAL_TEXT_SHORTCUT_DETECTED`) y nunca emite `BLOCK` autónomo sin evidencia visual.
2. **Mapas de Calor Grad-CAM:** Visualizamos las activaciones sobre `layer4` para confirmar que la atención espacial se concentra en elementos semánticos de la imagen.

In [ ]:
# Prueba de Red Teaming Sintético sobre casos positivos de Dev
if weights_file.exists():
    toxic_indices = [i for i, r in enumerate(dev_records) if r['label'] == 1][:50]
    asr_leaks = 0
    for idx in toxic_indices:
        sample = dev_ds[idx]
        zero_img = torch.zeros_like(sample['image_feat']).unsqueeze(0).to(device)
        res = router.route(zero_img, sample['text'].unsqueeze(0).to(device))
        if res['action'] == 'BLOCK':
            asr_leaks += 1
    print(f'Attack Success Rate (Bloqueos autónomos sin imagen): {asr_leaks}/{len(toxic_indices)} ({asr_leaks/len(toxic_indices):.1%})')

## Fase 5: Generación de Gráficas Vectoriales para el Paper (300 DPI)
Exporta directamente las figuras en `paper/figures/` listas para ser compiladas en `multimodal_trust_safety_paper.tex`.

In [ ]:
# Ejemplo de generación de figura comparativa de ablación
if summary_path.exists():
    fig, ax = plt.subplots(figsize=(7, 4), dpi=300)
    df_ablation[['auroc_global', 'fpr_benign_confounders']].plot(kind='bar', ax=ax, color=['#1f4e78', '#c00000'])
    ax.set_title('Global AUROC vs. Benign Confounder FPR across Fusion Architectures')
    ax.set_ylabel('Score / Rate')
    ax.set_xticklabels(df_ablation.index, rotation=25, ha='right')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    Path('paper/figures').mkdir(parents=True, exist_ok=True)
    plt.savefig('paper/figures/ablation_comparison.pdf')
    plt.show()